# 不规则计算场景

SIMT 更便于直接表达的三类典型场景——离散索引、散射写入、共享计数——正是 SIMT 的主场。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. Embedding Gather（离散索引读取）

`output[row, col] = input[indices[0, row], col]`——输出行号连续，但**读哪一行由索引表在运行时决定**：


In [ ]:
INPUT_ROWS, WIDTH, OUTPUT_ROWS = 1024, 64, 512
THREADS, BLOCKS = 256, 2

@pl.vector_function(mode="simt", max_threads=THREADS)
def gather_rows(output, input_tensor, indices, row_count: pl.DT_UINT32):
    first_row = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
    row_stride = pl.simt.grid_dim().x * pl.simt.block_dim().x
    for row in pl.range(first_row, row_count, row_stride):
        input_row = indices[0, row]                  # 运行时决定读哪一行
        for col in pl.range(0, WIDTH, 1):
            output[row, col] = input_tensor[input_row, col]


@pl.jit(auto_mutex=True)
def gather_kernel(
    input_tensor: pl.Tensor[[INPUT_ROWS, WIDTH], pl.DT_FP32],
    indices: pl.Tensor[[1, OUTPUT_ROWS], pl.DT_INT32],
    output: pl.Tensor[[OUTPUT_ROWS, WIDTH], pl.DT_FP32],
    row_count: pl.DT_UINT32,
):
    with pl.section_vector():
        gather_rows[THREADS](output, input_tensor, indices, row_count)

device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
input_tensor = torch.randn((INPUT_ROWS, WIDTH), dtype=torch.float32, device=device)
indices = torch.randint(INPUT_ROWS, (1, OUTPUT_ROWS), dtype=torch.int32, device=device)
output = torch.empty((OUTPUT_ROWS, WIDTH), dtype=torch.float32, device=device)
gather_kernel[None, BLOCKS](input_tensor, indices, output, OUTPUT_ROWS)
torch.npu.synchronize()
expected = input_tensor.index_select(0, indices.reshape(-1).to(torch.int64))
torch.testing.assert_close(output, expected, rtol=0, atol=0)

【待 950 验证：依据官方 Gather 示例缩小规模，补齐本地 Kernel 与 Host，不依赖 gym】

三个工程细节：

1. **调用方保证索引有效**：`0 ≤ indices[i] < INPUT_ROWS`，且 `0 ≤ row_count ≤ OUTPUT_ROWS`——动态下标前端不检查（第 6 章）；
2. **各线程写不同输出行**——行间无依赖，不需要屏障；
3. 行内用 `pl.range(0, WIDTH, 1)` 循环复制：**每线程一个输出行**的映射让索引计算最简。


## 2. Scatter（散射写入）

Gather 的镜像：**写哪里由索引决定**（`output[indices[i], :] = input[i, :]`）。风险点：

- **索引重复**：两个线程写同一目标行，结果取决于执行顺序——语义上就未定义；需要去重或改用原子合并（如取最大值）；
- 边界保护同 Gather：写前判断 `indices[i]` 在有效范围内。

【待 950 验证：Scatter 为课程自研示例。注意语义层级——官方另有 tile 级 `pl.scatter`（scatter_gather 类，片上 Tile 间按索引重排），本节推导的是 **GM 级散射**（按运行时索引写多个不连续位置），两者是不同的场景与接口（索引见 03.03 扩展接口面）。】


## 3. 直方图（共享计数）

`hist[bucket(x[i])] += 1`——**所有线程竞争同一组计数桶**，是原子操作的教科书场景：

```python
@pl.vector_function(mode="simt", max_threads=256)
def histogram(data, hist, count: pl.DT_UINT32, num_buckets: pl.DT_UINT32):
    first = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
    stride = pl.simt.grid_dim().x * pl.simt.block_dim().x
    for i in pl.range(first, count, stride):
        b = bucket_of(data[0, i])            # 标量计算得到桶号
        if 0 <= b and b < num_buckets:
            pl.simt.atomic_add(hist, [0, b], 1)  # 原子累加，避免丢更新
```

【实现骨架】bucket_of 尚需按数值分桶规则实现；Host 先将 INT32 hist 清零，并保证 num_buckets 不超过分配长度。多核更新前不能在各 Block 内分别清零共享 GM 桶。


## 4. 三场景的共性

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">场景</th><th style="text-align:left;">不规则性</th><th style="text-align:left;">关键机制</th><th style="text-align:left;">Tile 路径的比较范围</th></tr>
<tr><td>Gather</td><td>读地址运行时决定</td><td>索引映射 + grid-stride</td><td>Tile gather 支持片上索引重排；这里比较的是 GM 任意行访问</td></tr>
<tr><td>Scatter</td><td>写地址运行时决定</td><td>索引映射 + 重复处理</td><td>Tile scatter 与 GM 散射范围不同，重复写都需定义合并语义</td></tr>
<tr><td>直方图（数值分桶频数）</td><td>共享位置竞争</td><td>原子操作</td><td>桶号随数据值变，规约接口覆盖不了（tile histogram 仅覆盖字节累计语义）</td></tr>
</table>

**读不规则 → Gather；写不规则 → Scatter（慎重复索引）；共享计数 → 原子**。

> 语义提醒：官方 tile 级 `pl.histogram` 做的是**字节值 256 桶累计计数**（基数排序场景），与本表的"数值分桶频数"是两种语义——频数计数（每桶累加 1、桶号随数据值变）仍以 SIMT 原子为自然表达，tile histogram 不覆盖该场景；两者不是替代关系。


## 5. grid-stride 完整走读：从需求到循环

需求：处理 [1, 10000] 的数据，8 核 × 256 线程 = 2048 线程。逐线程一元素的设计只能覆盖 2048 个——**剩下的 7952 个怎么办？**

**思路推演**：

```text
朴素方案：每线程 while 循环 i += 2048 → 就是 grid-stride，但要自己管边界
Pro 写法：pl.range(first, count, stride) —— range 自带边界截断
```

**完整骨架**（对照第 6 章 06.03 的索引公式）：

```python
@pl.vector_function(mode="simt", max_threads=256)
def process_all(x, out, count: pl.DT_UINT32):
    # ① 全局编号：本线程在线程"宇宙"中的位置
    tid = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
    # ② 总线程数：stride 的来源
    stride = pl.simt.grid_dim().x * pl.simt.block_dim().x
    # ③ 跨步认领：tid, tid+stride, tid+2*stride, ... 直到 count
    for i in pl.range(tid, count, stride):
        out[0, i] = transform(x[0, i])
```

**为什么 stride 用全局线程总数（而不是核数或块内线程数）？**

- 保持 first=全局线程编号而把 stride 改为核数（8）：**会重复认领元素**，例如线程 0 的第二次访问 i=8 与线程 8 的首次访问重叠；这是正确性错误；
- stride = 全局线程数（2048）：相邻线程处理相邻元素（tid 与 tid+1）——Warp 内地址连续，通常有利于访存合并，但实际效率取决于布局与对齐。

**正确性论证**（每个 grid-stride 写完自问）：任意元素 i ∈ [0, count)，是否存在唯一的 (线程 t, 轮次 k) 使 t + k×2048 = i？—— i mod 2048 唯一确定 t，i div 2048 确定 k。**不重不漏**。


### 重复目标的最小仲裁例子

四个来源行的目标是 [2,2,5,2]，分数为 [0.8,0.8,-1.0,0.7]。目标 2 的最大分数为 0.8，同分取来源 0；目标 5 取来源 2；其他目标保持零。不能让“最后完成写入的线程”决定赢家，否则相同输入可能得到不同输出。

第一阶段只求每个目标的 best_score；第二阶段在最终 best_score 下求最小来源 ID；第三阶段只有该唯一来源复制整行。分数与 ID 分别更新却没有阶段边界，会读到尚未完成的最大分数。下面的独立参考将三个阶段拆成同 Stream 的三个 Kernel，并直接检查 winner 数组和最终值。


### 可执行的数值分桶例子

这里输入直接是 INT32 桶号，合法范围为 [0,16)，负数与 16 以上均忽略；输出是 16 桶频数，不是累计直方图。Host 在所有核启动前统一清零，Kernel 内只有原子加，不在每个 Block 再次清零。完整 Host 与 `torch.bincount` 对照见章节参考源码。


In [ ]:
@pl.vector_function(mode="simt",max_threads=256)
def histogram_threads(data,hist):
    first=pl.simt.block_idx().x*pl.simt.block_dim().x+pl.simt.thread_idx().x
    stride=pl.simt.grid_dim().x*pl.simt.block_dim().x
    for i in pl.range(first,1024,stride):
        bucket=data[0,i]
        if bucket >= 0 and bucket < 16:
            pl.simt.atomic_add(hist, [0,bucket], 1)


In [ ]:
@pl.jit()
def histogram_kernel(data: pl.Tensor[[1,1024],pl.DT_INT32],hist: pl.Tensor[[1,16],pl.DT_INT32]):
    with pl.section_vector():
        histogram_threads[256](data,hist)


## 6. 课后练习

1. （选择）Gather 示例中"各线程写不同输出行，不需要屏障"的原因：
   A. 线程天然同步<br/>
   B. 写目标互不相交，无数据依赖<br/>
   C. 原子操作替代了屏障
2. （判断）Scatter 中两个线程的索引指向同一目标行是安全的。（对/错）
3. （实践）直方图若改用"读取后写回"（`hist[0,b] = hist[0,b] + 1`）会发生什么？构造一个最小复现条件（多少线程、什么分布以及怎样的读写交错会丢更新）。

In [ ]:
!cat ./answer/07.03_answer.txt

## 7. 本节小结

- 三大不规则场景：**Gather（读）/ Scatter（写）/ 直方图（计数）**；
- Gather 骨架 = grid-stride + 运行时索引 + 行内循环；
- Scatter 的索引重复 = 未定义语义，需去重或原子合并；
- 直方图 = 原子操作的教科书；丢更新的复现条件是"两线程同时读到旧值"。